<a href="https://colab.research.google.com/github/IKKNIGHT/Protein_interactions/blob/main/Phase2_and_2b_GNN_Combined.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2 + 2b (v3): leakage-safe training, a stronger GNN, and honest baselines
**What changed to raise accuracy** (each is a real modelling change, not a tuning trick):
1. **Richer node features (31 vs 5):** multiscale shape index, screened electrostatic potential from side-chain charges, local hydrophobicity, atom-density burial, backbone flag, and residue type of the nearest atom.
2. **Better patch graph:** Gaussian distance-weighted edges; each node also gets its distance to the patch centre and its normal alignment with the centre.
3. **Better readout:** centre-node embedding + mean + max (the label belongs to the centre vertex; the old mean-pool diluted it).
4. **Bigger residual GCN** (4 layers, 128 wide) and **3-seed ensembling**.
5. **More training data** (up to ~3000 complexes, still excluding anything >=30% identical to a benchmark protein).
6. **Residue-level post-processing** (aggregation + spatial smoothing) **chosen on held-out validation surfaces, not on the benchmark**.

**Phase 2b (is the GNN actually adding anything?):** residue-level baselines trained on the same data (composition+burial; plus averaged surface features with no message passing), a spatial-null permutation test per target, a held-out-complex comparison (hundreds of surfaces, not 20 targets), and a feature ablation.

Time: roughly 1.5 h on an A100 with default settings (see `N_SEEDS`, `RUN_ABLATION`).

In [ ]:
# ── Cell 1: Mount, paths, GPU ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, time, random, subprocess, re, shutil, math
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'biopython', 'requests', 'scipy', 'scikit-learn', 'pandas'])

BASE = '/content/drive/MyDrive/Protein_interactions'
CONFIG, GOLD, RESULTS, MANIFEST = (f'{BASE}/{d}' for d in ('config', 'gold', 'results', 'manifests'))
GNN_DIR = f'{BASE}/gnn'
for d in (GNN_DIR, f'{GNN_DIR}/models', f'{GNN_DIR}/splits', f'{RESULTS}/gnn_scores_v3'):
    os.makedirs(d, exist_ok=True)
assert os.path.isdir(GOLD), 'gold/ not found - run the Phase 0/1 notebook first'

LOCAL = '/content/ppi_local'
RAW, FEAT = f'{LOCAL}/raw', f'{LOCAL}/feats_v3'
for d in (RAW, FEAT): os.makedirs(d, exist_ok=True)

SEED = 42
random.seed(SEED)
import numpy as np, pandas as pd, torch
np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch', torch.__version__, '| device:', device, '| CPUs:', os.cpu_count())

Mounted at /content/drive
torch 2.11.0+cu130 | device: cuda | CPUs: 12


In [ ]:
# ── Cell 2: RCSB helpers + benchmark exclusion via RCSB SEQUENCE SEARCH ───────
# v1 relied on RCSB cluster ids, which came back empty (0 clusters excluded), so nothing was excluded.
# Now: every benchmark protein sequence is searched against the PDB (>=30% identity, e-value<=1) and
# every entry containing a hit is removed from the training pool.
import requests
GQL, SEARCH = 'https://data.rcsb.org/graphql', 'https://search.rcsb.org/rcsbsearch/v2/query'

def fetch_meta(ids):
    out = {}
    for i in range(0, len(ids), 50):
        chunk = list(ids[i:i + 50])
        q = ('{ entries(entry_ids: %s){ rcsb_id polymer_entities{ rcsb_polymer_entity{pdbx_description}'
             ' entity_poly{rcsb_entity_polymer_type pdbx_seq_one_letter_code_can}'
             ' rcsb_polymer_entity_container_identifiers{entity_id auth_asym_ids} } } }') % json.dumps(chunk)
        r = requests.post(GQL, json={'query': q}, timeout=180); r.raise_for_status()
        for e in (r.json().get('data') or {}).get('entries') or []:
            ents = []
            for pe in e['polymer_entities'] or []:
                ep = pe['entity_poly'] or {}
                if 'protein' not in (ep.get('rcsb_entity_polymer_type') or '').lower(): continue
                ids_ = pe['rcsb_polymer_entity_container_identifiers']
                ents.append({'entity_id': ids_['entity_id'], 'chains': ids_['auth_asym_ids'],
                             'seq': ep.get('pdbx_seq_one_letter_code_can') or '',
                             'desc': (pe['rcsb_polymer_entity'] or {}).get('pdbx_description') or ''})
            out[e['rcsb_id']] = ents
    return out

def homolog_entries(seq, max_rows=30000):
    found, start, total = set(), 0, None
    while total is None or start < min(total, max_rows):
        q = {"query": {"type": "terminal", "service": "sequence",
                       "parameters": {"evalue_cutoff": 1, "identity_cutoff": 0.3, "sequence_type": "protein", "value": seq}},
             "return_type": "polymer_entity", "request_options": {"paginate": {"start": start, "rows": 10000}}}
        r = requests.post(SEARCH, json=q, timeout=300)
        if r.status_code == 204: break
        r.raise_for_status()
        j = r.json(); total = j.get('total_count', 0)
        found |= {h['identifier'].split('_')[0] for h in j.get('result_set', [])}
        start += 10000
    return found

assign = json.load(open(f'{CONFIG}/chain_assignments.json'))
bench = {k: v for k, v in assign.items() if v['status'] in ('AUTO', 'CONFIRMED')}
bench_pdbs = sorted({v['pdb_id'] for v in bench.values()})
bm = fetch_meta(bench_pdbs)

EXCL_PATH = f'{CONFIG}/exclusion_entries_v2.json'
if os.path.exists(EXCL_PATH):
    EXCLUDED = set(json.load(open(EXCL_PATH))['entries']); print('Loaded cached exclusion set:', len(EXCLUDED))
else:
    uniq = {}
    for pdb, ents in bm.items():
        for e in ents: uniq.setdefault(e['seq'], f'{pdb}_{e["entity_id"]}')
    EXCLUDED, skipped = set(bench_pdbs), []
    for n, (seq, label) in enumerate(uniq.items(), 1):
        if len(seq) < 25:                                  # too short for the sequence-search service
            skipped.append(label); continue
        try:
            hits = homolog_entries(seq); EXCLUDED |= hits
            print(f'  [{n}/{len(uniq)}] {label} ({len(seq)} aa): {len(hits)} homologous entries', flush=True)
        except Exception as ex:
            skipped.append(label); print(f'  [{n}/{len(uniq)}] {label}: search failed ({ex})')
    json.dump({'entries': sorted(EXCLUDED), 'skipped_short_or_failed': skipped}, open(EXCL_PATH, 'w'))
    print('skipped (short peptides / failures):', skipped)

# Sanity guard: if the search silently failed the set would be tiny.
print(f'{len(EXCLUDED)} PDB entries excluded (contain a protein >=30% identical to a benchmark protein)')
assert len(EXCLUDED) > 5 * len(bench_pdbs), 'exclusion set suspiciously small - sequence search probably failed'

Loaded cached exclusion set: 8657
8657 PDB entries excluded (contain a protein >=30% identical to a benchmark protein)


In [ ]:
# ── Cell 3: Build the training pool (RCSB search -> exclude -> group/dedupe -> grouped split) ──
N_COMPLEXES   = 3000   # cap on training complexes (featurising 500 took 1.5 min, so this is cheap)
MAX_PER_GROUP = 6      # stops lysozyme / antibody / protease complexes dominating
POOL_SAMPLE   = 12000   # entries sampled before filtering

import re, random
from collections import Counter, defaultdict

GENERIC = {'protein', 'peptide', 'unknown', 'unnamed protein product', 'uncharacterized protein',
           'hypothetical protein', 'fab', 'heavy chain', 'light chain'}

def kmers(seq, k=5):
    return {seq[i:i + k] for i in range(len(seq) - k + 1)}

def group_entities(ents, kmer=4, frac=0.12, max_posting=200):
    # Cheap redundancy grouping for the TRAINING pool only (shared 4-mers catch >~60% identity, and equal
    # names are merged). The benchmark exclusion does NOT use this; it uses RCSB's sequence search.
    parent = {}
    def find(x):
        parent.setdefault(x, x)
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb: parent[ra] = rb
    index, sets, by_desc = defaultdict(list), {}, {}
    for e in ents:
        k = e['key']; find(k)
        ks = kmers(e['seq'], kmer); sets[k] = ks
        cnt = Counter()
        for km in ks:
            post = index.get(km)
            if post and len(post) <= max_posting: cnt.update(post)
        for o, c in cnt.items():
            if c >= frac * min(len(ks), len(sets[o])): union(k, o)
        for km in ks: index[km].append(k)
        d = re.sub(r'\s+', ' ', (e.get('desc') or '').lower().strip())
        if len(d) >= 6 and d not in GENERIC:
            if d in by_desc: union(k, by_desc[d])
            else: by_desc[d] = k
    return {e['key']: find(e['key']) for e in ents}

def cap_and_dedupe(cands, max_per_group=3, limit=None):
    seen_pair, per_group, keep = set(), Counter(), []
    for c in cands:
        pair = tuple(sorted(c['groups']))
        if pair in seen_pair: continue
        if any(per_group[g] >= max_per_group for g in set(c['groups'])): continue
        seen_pair.add(pair)
        for g in set(c['groups']): per_group[g] += 1
        keep.append(c)
        if limit and len(keep) >= limit: break
    return keep

def group_split(keep, frac_val=0.15, frac_test=0.15, seed=42):
    parent = {}
    def find(x):
        parent.setdefault(x, x)
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for c in keep:
        a, b = find(c['groups'][0]), find(c['groups'][1])
        if a != b: parent[a] = b
    comps = {}
    for c in keep: comps.setdefault(find(c['groups'][0]), []).append(c)
    keys = sorted(comps); random.Random(seed).shuffle(keys)
    n, nv, nt = len(keep), 0, 0
    for k in keys:
        if len(comps[k]) > 0.3 * n: s = 'train'
        elif nt < frac_test * n: s = 'test'; nt += len(comps[k])
        elif nv < frac_val * n: s = 'val'; nv += len(comps[k])
        else: s = 'train'
        for c in comps[k]: c['split'] = s
    return keep

def txt(attr, op, val):
    return {"type": "terminal", "service": "text", "parameters": {"attribute": attr, "operator": op, "value": val}}

query = {"query": {"type": "group", "logical_operator": "and", "nodes": [
            txt("exptl.method", "exact_match", "X-RAY DIFFRACTION"),
            txt("rcsb_entry_info.resolution_combined", "less_or_equal", 2.5),
            txt("rcsb_entry_info.polymer_entity_count_protein", "equals", 2),
            txt("rcsb_entry_info.deposited_polymer_entity_instance_count", "equals", 2),
            txt("rcsb_entry_info.polymer_entity_count_DNA", "equals", 0),
            txt("rcsb_entry_info.polymer_entity_count_RNA", "equals", 0)]},
         "return_type": "entry", "request_options": {"paginate": {"start": 0, "rows": 10000}}}

POOL_PATH = f'{GNN_DIR}/splits/training_pool_v3.json'
if os.path.exists(POOL_PATH):
    pool = json.load(open(POOL_PATH)); print('Loaded existing pool:', len(pool['complexes']), 'complexes')
else:
    hits, start, total = [], 0, None
    while total is None or start < min(total, 30000):
        query['request_options']['paginate']['start'] = start
        r = requests.post(SEARCH, json=query, timeout=180)
        if r.status_code != 200:
            raise RuntimeError(f'RCSB search failed ({r.status_code}): {r.text[:500]}')
        j = r.json(); total = j.get('total_count', 0)
        hits += [h['identifier'] for h in j.get('result_set', [])]; start += 10000
    print('search hits:', len(hits), '(total', total, ')')
    random.Random(SEED).shuffle(hits)
    pre = hits[:POOL_SAMPLE]
    hits = [h for h in pre if h not in EXCLUDED]
    print(f'removed {len(pre) - len(hits)} of {len(pre)} sampled entries as homologous to benchmark proteins')
    meta = fetch_meta(hits)

    ents, cands = [], []
    for pdb in hits:
        e2 = meta.get(pdb, [])
        if len(e2) != 2 or any(len(e['chains']) != 1 or len(e['seq']) < 20 for e in e2): continue
        for i, e in enumerate(e2): ents.append({'key': f'{pdb}_{i}', 'seq': e['seq'], 'desc': e['desc']})
        cands.append({'pdb': pdb, 'chains': [e['chains'][0] for e in e2]})
    grp = group_entities(ents)
    for c in cands: c['groups'] = [grp[f'{c["pdb"]}_0'], grp[f'{c["pdb"]}_1']]
    print('candidates after shape filter:', len(cands), '| redundancy groups:', len(set(grp.values())))
    keep = group_split(cap_and_dedupe(cands, MAX_PER_GROUP, N_COMPLEXES))
    pool = {'complexes': keep}
    json.dump(pool, open(POOL_PATH, 'w'))

from collections import Counter
print('complexes:', len(pool['complexes']), dict(Counter(c['split'] for c in pool['complexes'])))

Loaded existing pool: 888 complexes
complexes: 888 {'test': 134, 'train': 620, 'val': 134}


In [ ]:
# ── Cell 4: Surface geometry + 31 node features (with self-tests) ─────────────
import numpy as np
from scipy.spatial import cKDTree

VDW = {'C': 1.70, 'N': 1.55, 'O': 1.52, 'S': 1.80, 'P': 1.80, 'SE': 1.90}

KD_RAW = {'ILE': 4.5, 'VAL': 4.2, 'LEU': 3.8, 'PHE': 2.8, 'CYS': 2.5, 'MET': 1.9, 'ALA': 1.8, 'GLY': -0.4,
          'THR': -0.7, 'SER': -0.8, 'TRP': -0.9, 'TYR': -1.3, 'PRO': -1.6, 'HIS': -3.2, 'GLU': -3.5,
          'GLN': -3.5, 'ASP': -3.5, 'ASN': -3.5, 'LYS': -3.9, 'ARG': -4.5}
KD_SCALE = {k: (v + 4.5) / 9.0 * 2 - 1 for k, v in KD_RAW.items()}      # [-1, 1]

CHARGE_ATOMS = {'N': 0.41, 'NZ': 0.34, 'NH1': 0.46, 'NH2': 0.46, 'ND1': 0.36, 'NE2': 0.31, 'NE': 0.34,
                'OG': -0.66, 'OG1': -0.67, 'OH': -0.61, 'O': -0.57, 'OD1': -0.62, 'OD2': -0.62,
                'OE1': -0.60, 'OE2': -0.60, 'OXT': -0.80, 'SD': -0.09}
HBOND_DONORS    = {'N', 'NZ', 'NH1', 'NH2', 'ND1', 'NE2', 'NE', 'OG', 'OG1', 'OH'}
HBOND_ACCEPTORS = {'O', 'OD1', 'OD2', 'OE1', 'OE2', 'OXT', 'SD', 'ND1', 'NE2', 'OH'}


def fibonacci_sphere(n):
    i = np.arange(n) + 0.5
    phi = np.arccos(1 - 2 * i / n)
    theta = np.pi * (1 + 5 ** 0.5) * i
    return np.stack([np.cos(theta) * np.sin(phi), np.sin(theta) * np.sin(phi), np.cos(phi)], 1)


def sas_points(coords, radii, probe=1.4, n_pts=100, voxel=2.0):
    # Solvent-accessible surface points: points on each atom's (radius + probe) sphere that lie inside
    # NO other atom's (radius + probe) sphere. Every neighbour is checked (the old notebook checked only
    # the single nearest atom). Points are then merged on a voxel grid so spacing is ~voxel Angstrom.
    coords = np.asarray(coords, float); radii = np.asarray(radii, float)
    ext = radii + probe
    sph = fibonacci_sphere(n_pts)
    tree = cKDTree(coords)
    rmax = ext.max()
    P, N = [], []
    for i in range(len(coords)):
        p = coords[i] + ext[i] * sph
        nb = [j for j in tree.query_ball_point(coords[i], ext[i] + rmax) if j != i]
        if nb:
            d = np.linalg.norm(p[:, None, :] - coords[nb][None, :, :], axis=2)
            p = p[~(d < ext[nb][None, :] - 1e-3).any(axis=1)]
        if len(p):
            P.append(p); N.append((p - coords[i]) / ext[i])
    if not P:
        return np.zeros((0, 3)), np.zeros((0, 3))
    P = np.concatenate(P); N = np.concatenate(N)
    key = np.floor(P / voxel).astype(np.int64)
    key -= key.min(0)
    m = key.max(0) + 1
    flat = (key[:, 0] * m[1] + key[:, 1]) * m[2] + key[:, 2]
    _, inv = np.unique(flat, return_inverse=True)
    inv = inv.ravel()
    cnt = np.bincount(inv)
    V = np.zeros((len(cnt), 3)); Nn = np.zeros((len(cnt), 3))
    np.add.at(V, inv, P); np.add.at(Nn, inv, N)
    V /= cnt[:, None]
    Nn /= np.linalg.norm(Nn, axis=1, keepdims=True) + 1e-9
    return V, Nn


def shape_index(verts, normals, k=12):
    # Local quadric fit z = a x^2 + b xy + c y^2 in a tangent frame whose z axis is the OUTWARD normal
    # (the old code used an eigenvector of arbitrary sign, so convex/concave was randomly flipped).
    # Convention: convex cap -> +1, concave bowl -> -1, saddle ~ 0. Umbilic points give +-1, not 0.
    n_pts = len(verts)
    k = min(k, n_pts - 1)
    tree = cKDTree(verts)
    _, idx = tree.query(verts, k=k + 1)
    nb = verts[idx[:, 1:]] - verts[:, None, :]
    n = normals / (np.linalg.norm(normals, axis=1, keepdims=True) + 1e-9)
    ref = np.where(np.abs(n[:, 2:3]) < 0.9, np.array([[0., 0., 1.]]), np.array([[1., 0., 0.]]))
    u = np.cross(n, ref); u /= np.linalg.norm(u, axis=1, keepdims=True)
    w = np.cross(n, u)
    x = (nb * u[:, None, :]).sum(2); y = (nb * w[:, None, :]).sum(2); z = (nb * n[:, None, :]).sum(2)
    A = np.stack([x * x, x * y, y * y], axis=2)
    AtA = np.einsum('nki,nkj->nij', A, A) + 1e-6 * np.eye(3)
    Atz = np.einsum('nki,nk->ni', A, z)
    a, b, c = np.linalg.solve(AtA, Atz[..., None])[..., 0].T
    h11, h12, h22 = -2 * a, -b, -2 * c            # sign flip: convex w.r.t. outward normal is positive
    mean = (h11 + h22) / 2
    diff = np.sqrt(((h11 - h22) / 2) ** 2 + h12 ** 2)
    k1, k2 = mean + diff, mean - diff
    return ((2 / np.pi) * np.arctan2(k1 + k2, k1 - k2)).astype(np.float32)



AA20 = ['ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'GLN', 'GLU', 'GLY', 'HIS', 'ILE',
        'LEU', 'LYS', 'MET', 'PHE', 'PRO', 'SER', 'THR', 'TRP', 'TYR', 'VAL']
AA_IDX = {a: i for i, a in enumerate(AA20)}
BACKBONE = {'N', 'CA', 'C', 'O', 'OXT'}
SC_CHARGE = {('LYS', 'NZ'): 1.0, ('ARG', 'CZ'): 1.0, ('ASP', 'CG'): -1.0, ('GLU', 'CD'): -1.0}
RES_CHARGE = {'LYS': 1.0, 'ARG': 1.0, 'ASP': -1.0, 'GLU': -1.0}

FEATURE_NAMES = ['si', 'si_wide', 'phi', 'hydro', 'hbd', 'hba', 'elec', 'hydro_sm', 'burial', 'bb'] + \
                ['aa_' + a for a in AA20 + ['OTH']]
FEATURE_GROUPS = {'shape': [0, 1], 'atomic_chem': [2, 3, 4, 5, 9], 'electro_smooth': [6, 7],
                  'burial': [8], 'aa': list(range(10, 31))}
N_FEAT = len(FEATURE_NAMES)      # 31


def _pairs(q_xyz, a_xyz, r):
    M = cKDTree(q_xyz).sparse_distance_matrix(cKDTree(a_xyz), r, output_type='coo_matrix')
    return M.row, M.col, M.data


def vertex_features(V, N, a_xyz, a_res, a_name):
    # 31 per-vertex features: multiscale shape index, atomic chemistry, a screened electrostatic proxy from
    # side-chain charges, local hydrophobicity, atom-density burial, backbone flag, nearest-residue type.
    a_xyz = np.asarray(a_xyz)
    n = len(V)
    hyd = np.array([KD_SCALE.get(r, 0.0) for r in a_res], np.float32)
    chg = np.array([np.clip(CHARGE_ATOMS.get(m, 0.0) / 0.8, -1, 1) for m in a_name], np.float32)
    don = np.array([m in HBOND_DONORS for m in a_name], np.float32)
    acc = np.array([m in HBOND_ACCEPTORS for m in a_name], np.float32)
    q = np.array([SC_CHARGE.get((r, m), 0.0) for r, m in zip(a_res, a_name)], np.float32)
    aa = np.array([AA_IDX.get(r, 20) for r in a_res])
    bb = np.array([m in BACKBONE for m in a_name], np.float32)

    si = shape_index(V, N, 12)
    sw = shape_index(V, N, 30)
    d, near = cKDTree(a_xyz).query(V)
    ok = (d <= 3.6).astype(np.float32)
    wgt = np.clip(1 - d / 3.6, 0, 1).astype(np.float32) * ok
    rows, cols, dist = _pairs(V, a_xyz, 12.0)
    cnt12 = np.bincount(rows, minlength=n)
    elec = np.tanh(np.bincount(rows, weights=q[cols] * np.exp(-dist / 8.0), minlength=n) / 2.0)
    m8 = dist <= 8.0
    cnt8 = np.bincount(rows[m8], minlength=n)
    hs = np.bincount(rows[m8], weights=hyd[cols[m8]], minlength=n) / np.maximum(cnt8, 1)
    aa_oh = np.zeros((n, 21), np.float32); aa_oh[np.arange(n), aa[near]] = 1
    F = np.column_stack([si, sw, chg[near] * ok, hyd[near] * ok, don[near] * wgt, acc[near] * wgt,
                         elec, hs, cnt12 / 300.0, bb[near], aa_oh]).astype(np.float32)
    return F, near


def interface_labels(verts, partner_xyz, cutoff=5.0):
    d, _ = cKDTree(np.asarray(partner_xyz)).query(verts)
    return (d < cutoff).astype(np.int8)


def featurize_surface2(a_xyz, a_elem, a_res, a_name, partner_xyz=None):
    a_xyz = np.asarray(a_xyz)
    radii = np.array([VDW.get(e.upper(), 1.7) for e in a_elem])
    V, N = sas_points(a_xyz, radii)
    if len(V) < 200:
        return None
    F, near = vertex_features(V, N, a_xyz, a_res, a_name)
    out = {'verts': V.astype(np.float32), 'normals': N.astype(np.float32), 'feats': F, 'near': near}
    if partner_xyz is not None:
        out['labels'] = interface_labels(V, partner_xyz)
        out['atom_iface'] = interface_labels(a_xyz, partner_xyz).astype(bool)
    return out


def patch_arrays2(verts, normals, feats, tree, seed, size=80, k=9, sigma=3.0):
    # Patch = `size` vertices nearest the seed (seed is row 0). Node input = features + distance to seed
    # + normal alignment with seed (rotation invariant). Edges: kNN (k=9), Gaussian distance weights,
    # symmetrised, self loops, symmetric normalisation.
    _, idx = tree.query(verts[seed], k=size)
    pos = verts[idx]
    nrm = normals[idx]
    x = np.concatenate([feats[idx], (np.linalg.norm(pos - pos[0], axis=1) / 10.0)[:, None],
                        (nrm @ nrm[0])[:, None]], axis=1).astype(np.float32)
    d = np.linalg.norm(pos[:, None, :] - pos[None, :, :], axis=2)
    np.fill_diagonal(d, np.inf)
    nbr = np.argsort(d, axis=1)[:, :k]
    rows = np.repeat(np.arange(size), k); cols = nbr.ravel()
    A = np.zeros((size, size), np.float32)
    A[rows, cols] = np.exp(-(d[rows, cols] / sigma) ** 2)
    A = np.maximum(A, A.T) + np.eye(size, dtype=np.float32)
    dinv = 1 / np.sqrt(A.sum(1))
    return x, (A * dinv[:, None] * dinv[None, :]).astype(np.float32)


# ---------------- residue-level helpers (shared by training data, baselines and benchmark) ----------------
def residue_geometry(a_xyz, a_resid, n_res):
    a_xyz = np.asarray(a_xyz)
    cnt = np.bincount(a_resid, minlength=n_res).astype(float)
    cen = np.column_stack([np.bincount(a_resid, weights=a_xyz[:, i], minlength=n_res) for i in range(3)])
    return cen / np.maximum(cnt, 1)[:, None]


def residue_table(a_xyz, a_resid, res_names, near, F, n_res):
    # Per-residue features for the baselines. B1 = composition/burial only (no surface geometry).
    # B2 = B1 + vertex-averaged surface features (what a GNN sees, but with no message passing).
    a_xyz = np.asarray(a_xyz)
    cen = residue_geometry(a_xyz, a_resid, n_res)
    vres = a_resid[near]
    n_vert = np.bincount(vres, minlength=n_res)
    burial = np.array(cKDTree(a_xyz).query_ball_point(cen, 10.0, return_length=True), float)
    kd = np.array([KD_SCALE.get(r, 0.0) for r in res_names], np.float32)
    ch = np.array([RES_CHARGE.get(r, 0.0) for r in res_names], np.float32)
    nb = cKDTree(cen).query_ball_point(cen, 8.0)
    nb_h = np.array([kd[j].mean() for j in nb]); nb_c = np.array([ch[j].sum() for j in nb])
    nb_n = np.array([len(j) for j in nb], float)
    aa = np.zeros((n_res, 21), np.float32)
    aa[np.arange(n_res), [AA_IDX.get(r, 20) for r in res_names]] = 1
    B1 = np.column_stack([aa, n_vert, burial / 100.0, kd, ch, nb_h, nb_c, nb_n])
    gcols = [0, 1, 2, 3, 4, 5, 6, 7, 8]                         # shape, atomic chem, electro, smooth, burial
    S = np.column_stack([np.bincount(vres, weights=F[:, c], minlength=n_res) for c in gcols])
    S = S / np.maximum(n_vert, 1)[:, None]
    mx = np.full(n_res, -1.0)
    np.maximum.at(mx, vres, F[:, 0])
    B2 = np.column_stack([B1, S, mx])
    return {'B1': B1.astype(np.float32), 'B2': B2.astype(np.float32), 'n_vert': n_vert, 'centroid': cen}


def aggregate_vertices(vscore, near, a_resid, n_res, how='top3'):
    vres = a_resid[near]
    order = np.argsort(vres, kind='stable')
    vs, rs_ = vscore[order], vres[order]
    bounds = np.searchsorted(rs_, np.arange(n_res + 1))
    out = np.zeros(n_res, np.float64)
    for r in range(n_res):
        seg = vs[bounds[r]:bounds[r + 1]]
        if len(seg) == 0: continue
        out[r] = {'top3': lambda s: np.sort(s)[-3:].mean(), 'mean': np.mean, 'max': np.max}[how](seg)
    return out


def smooth_residues(scores, centroid, alpha, radius=8.0):
    if alpha <= 0: return scores
    nb = cKDTree(centroid).query_ball_point(centroid, radius)
    nbm = np.array([scores[j].mean() for j in nb])
    return (1 - alpha) * scores + alpha * nbm


def residue_labels(a_resid, atom_iface, n_res):
    return (np.bincount(a_resid, weights=atom_iface.astype(float), minlength=n_res) > 0).astype(int)


# ---- self-tests: must pass before anything is trained ----
_R = 8.0; _s = fibonacci_sphere(3000) * _R
assert shape_index(_s, _s / _R).mean() > 0.9 and shape_index(_s, -_s / _R).mean() < -0.9, 'shape index sign'
_rng = np.random.default_rng(0)
_g = np.arange(-9, 9.1, 1.9); _G = np.stack(np.meshgrid(_g, _g, _g), -1).reshape(-1, 3)
_G = _G[np.linalg.norm(_G, axis=1) < 10] + _rng.normal(0, 0.25, (int((np.linalg.norm(_G, axis=1) < 10).sum()), 3))
_nres = len(_G) // 6; _G = _G[:_nres * 6]; _rid = np.repeat(np.arange(_nres), 6)
_rn = list(_rng.choice(AA20, _nres)); _nm = list(np.tile(['N', 'CA', 'C', 'O', 'CB', 'NZ'], _nres))
_o = featurize_surface2(_G, ['C'] * len(_G), [_rn[i] for i in _rid], _nm, _G[:60] + np.array([0, 0, 6.0]))
assert _o['feats'].shape[1] == N_FEAT and np.isfinite(_o['feats']).all()
_x, _A = patch_arrays2(_o['verts'], _o['normals'], _o['feats'], cKDTree(_o['verts']), 3)
assert _x.shape == (80, N_FEAT + 2) and np.allclose(_A, _A.T) and np.linalg.eigvalsh(_A).max() < 1.0001
_rt = residue_table(_G, _rid, _rn, _o['near'], _o['feats'], _nres)
assert np.isfinite(_rt['B2']).all()
print(f'Geometry self-tests passed ({N_FEAT} features, patch input dim {N_FEAT + 2}).')

Geometry self-tests passed (31 features, patch input dim 33).


In [ ]:
# ── Cell 5: Featurise training complexes (parallel, resumable) ────────────────
from Bio import PDB
from multiprocessing import Pool

def download_cif(pdb):
    path = f'{RAW}/{pdb}.cif'
    if not os.path.exists(path):
        r = requests.get(f'https://files.rcsb.org/download/{pdb}.cif', timeout=120); r.raise_for_status()
        open(path, 'wb').write(r.content)
    return path

def chain_atoms(chain):
    xyz, el, rs, nm, rid, rnames = [], [], [], [], [], []
    for r in chain:
        rn = r.get_resname()
        if r.id[0] != ' ' and rn != 'MSE': continue          # keep selenomethionine, treated as MET
        rn = 'MET' if rn == 'MSE' else rn
        added = False
        for a in r:
            e = (a.element or a.get_name()[0]).strip().upper()
            if e == 'H' or a.get_altloc() not in (' ', 'A'): continue
            xyz.append(a.coord); el.append(e); rs.append(rn); nm.append(a.get_name().strip()); rid.append(len(rnames))
            added = True
        if added: rnames.append(rn)
    return np.array(xyz), el, rs, nm, np.array(rid, np.int32), rnames

def job(args):
    pdb, ca, cb = args
    outs = [f'{FEAT}/{pdb}_{ca}.npz', f'{FEAT}/{pdb}_{cb}.npz']
    if all(os.path.exists(o) for o in outs): return pdb, 'cached'
    try:
        model = PDB.MMCIFParser(QUIET=True).get_structure(pdb, download_cif(pdb))[0]
        A, B = chain_atoms(model[ca]), chain_atoms(model[cb])
        for (me, other), out in zip(((A, B), (B, A)), outs):
            if len(me[0]) < 50 or len(other[0]) < 20: raise ValueError('chain too small')
            res = featurize_surface2(me[0], me[1], me[2], me[3], partner_xyz=other[0])
            if res is None: raise ValueError('too few surface vertices')
            if res['labels'].sum() < 20: raise ValueError('interface too small')
            np.savez_compressed(out, verts=res['verts'], normals=res['normals'].astype(np.float16),
                                feats=res['feats'].astype(np.float16), labels=res['labels'], near=res['near'].astype(np.int32),
                                atom_xyz=me[0].astype(np.float32), atom_resid=me[4], atom_iface=res['atom_iface'],
                                res_names=np.array(me[5], dtype='U3'))
        return pdb, 'ok'
    except Exception as e:
        return pdb, f'fail: {e}'

ARCHIVE = f'{GNN_DIR}/features_cache_v3.tar'
if not os.listdir(FEAT) and os.path.exists(ARCHIVE):
    subprocess.run(['tar', '-xf', ARCHIVE, '-C', FEAT]); print('Restored feature cache from Drive')

jobs = [(c['pdb'], c['chains'][0], c['chains'][1]) for c in pool['complexes']]
t0 = time.time(); status = {}
with Pool(min(10, os.cpu_count() or 2)) as p:
    for i, (pdb, st) in enumerate(p.imap_unordered(job, jobs), 1):
        status[pdb] = st
        if i % 200 == 0: print(f'  {i}/{len(jobs)}  ({(time.time()-t0)/60:.1f} min)', flush=True)
print(dict(Counter(s.split(':')[0] for s in status.values())), f'in {(time.time()-t0)/60:.1f} min')
print('failure reasons:', dict(Counter(s for s in status.values() if s.startswith('fail'))))

subprocess.run(['tar', '-cf', ARCHIVE, '-C', FEAT, '.'])
print('Feature cache archived to Drive:', round(os.path.getsize(ARCHIVE) / 1e6, 1), 'MB')

splits = {}
for c in pool['complexes']:
    for ch in c['chains']:
        if os.path.exists(f'{FEAT}/{c["pdb"]}_{ch}.npz'):
            splits.setdefault(c['split'], []).append(f'{c["pdb"]}_{ch}')
print({k: len(v) for k, v in splits.items()}, 'surfaces per split')
json.dump(splits, open(f'{GNN_DIR}/splits/surface_splits_v3.json', 'w'))

  200/888  (0.0 min)
  400/888  (0.0 min)
  600/888  (0.0 min)
  800/888  (0.0 min)
{'cached': 868, 'fail': 20} in 0.0 min
failure reasons: {'fail: chain too small': 18, 'fail: interface too small': 2}
Feature cache archived to Drive: 180.0 MB
{'test': 260, 'train': 1217, 'val': 260} surfaces per split


In [ ]:
# ── Cell 6: Model, patch dataset, training function ───────────────────────────
import torch.nn as nn, torch.nn.functional as F_
from sklearn.metrics import roc_auc_score, average_precision_score

ALL_COLS = list(range(N_FEAT))

class PatchDS(torch.utils.data.Dataset):
    # Each epoch draws n_pos interface-centred and n_neg non-interface-centred patches per surface.
    def __init__(self, surface_ids, cols, n_pos=40, n_neg=40, seed=0):
        self.ids, self.cols, self.n_pos, self.n_neg, self.seed, self.epoch, self.S = \
            list(surface_ids), list(cols), n_pos, n_neg, seed, 0, {}
        for s in self.ids:
            z = np.load(f'{FEAT}/{s}.npz'); v, y = z['verts'], z['labels']
            self.S[s] = dict(verts=v, normals=z['normals'].astype(np.float32),
                             feats=z['feats'][:, self.cols].astype(np.float32), tree=cKDTree(v),
                             pos=np.where(y == 1)[0], neg=np.where(y == 0)[0])
        self.per = n_pos + n_neg
    def __len__(self): return len(self.ids) * self.per
    def __getitem__(self, i):
        S = self.S[self.ids[i // self.per]]; want = int(i % self.per < self.n_pos)
        pool_ = S['pos'] if want else S['neg']
        rng = np.random.default_rng([self.epoch, i, self.seed])
        x, A = patch_arrays2(S['verts'], S['normals'], S['feats'], S['tree'], pool_[rng.integers(len(pool_))])
        return torch.from_numpy(x), torch.from_numpy(A), torch.tensor(float(want))

class DenseGCN2(nn.Module):
    def __init__(self, in_dim, hidden=128, layers=4, p=0.3):
        super().__init__()
        self.inp = nn.Linear(in_dim, hidden)
        self.lins = nn.ModuleList(nn.Linear(hidden, hidden) for _ in range(layers))
        self.bns = nn.ModuleList(nn.BatchNorm1d(hidden) for _ in range(layers))
        self.head = nn.Sequential(nn.Linear(3 * hidden, 256), nn.ReLU(), nn.Dropout(p),
                                  nn.Linear(256, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x, A):                           # x [B,80,F]  A [B,80,80] (normalised); node 0 = centre
        h = F_.relu(self.inp(x))
        for lin, bn in zip(self.lins, self.bns):
            z = A @ lin(h); B, N, C = z.shape
            h = h + F_.relu(bn(z.reshape(B * N, C)).reshape(B, N, C))
        return self.head(torch.cat([h[:, 0], h.mean(1), h.max(1).values], 1)).squeeze(-1)

NUM_WORKERS = min(8, os.cpu_count() or 2)
loader = lambda ds, sh, bs=256: torch.utils.data.DataLoader(ds, batch_size=bs, shuffle=sh, num_workers=NUM_WORKERS)

@torch.no_grad()
def patch_eval(models, ds):
    for m in models: m.eval()
    ys, ps = [], []
    for x, A, y in loader(ds, False, 512):
        x, A = x.to(device), A.to(device)
        ps.append(torch.stack([torch.sigmoid(m(x, A)) for m in models]).mean(0).cpu().numpy()); ys.append(y.numpy())
    y, p = np.concatenate(ys), np.concatenate(ps)
    return float(roc_auc_score(y, p)), float(average_precision_score(y, p))

def train_model(cols, epochs, seed, tag, n_pos=40, n_neg=40, verbose=True):
    torch.manual_seed(seed); np.random.seed(seed)
    tr, va = PatchDS(splits['train'], cols, n_pos, n_neg, seed), PatchDS(splits['val'], cols, 24, 24, 0)
    model = DenseGCN2(len(cols) + 2).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best, hist, path = -1, [], f'{GNN_DIR}/models/gnn_v3_{tag}.pt'
    for ep in range(1, epochs + 1):
        tr.epoch = ep; model.train(); tl = n = 0
        for x, A, y in loader(tr, True):
            x, A, y = x.to(device), A.to(device), y.to(device)
            loss = F_.binary_cross_entropy_with_logits(model(x, A), y)
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tl += loss.item() * len(y); n += len(y)
        sched.step()
        auc, ap = patch_eval([model], va)
        hist.append({'epoch': ep, 'train_loss': tl / n, 'val_auroc': auc, 'val_auprc': ap})
        flag = ''
        if ap > best:
            best = ap; flag = ' <- best'; torch.save({'model': model.state_dict(), 'cols': list(cols), 'epoch': ep}, path)
        if verbose or ep == epochs:
            print(f'[{tag}] epoch {ep:2d} | train loss {tl/n:.4f} | val AUROC {auc:.3f} AUPRC {ap:.3f}{flag}', flush=True)
    model.load_state_dict(torch.load(path, map_location=device, weights_only=False)['model']); model.eval()
    return model, pd.DataFrame(hist)
print('Model parameters:', sum(p.numel() for p in DenseGCN2(N_FEAT + 2).parameters()))

Model parameters: 186497


In [ ]:
# ── Cell 6b: regularised model + early stopping (overrides Cell 6 definitions) ──
class DenseGCN2(nn.Module):
    def __init__(self, in_dim, hidden=96, layers=3, p=0.5, feat_drop=0.15):
        super().__init__()
        self.fdrop = nn.Dropout(feat_drop)
        self.inp = nn.Linear(in_dim, hidden)
        self.lins = nn.ModuleList(nn.Linear(hidden, hidden) for _ in range(layers))
        self.bns = nn.ModuleList(nn.BatchNorm1d(hidden) for _ in range(layers))
        self.head = nn.Sequential(nn.Linear(3 * hidden, 128), nn.ReLU(), nn.Dropout(p),
                                  nn.Linear(128, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x, A):
        h = F_.relu(self.inp(self.fdrop(x)))
        for lin, bn in zip(self.lins, self.bns):
            z = A @ lin(h); B, N, C = z.shape
            h = h + F_.relu(bn(z.reshape(B * N, C)).reshape(B, N, C))
        return self.head(torch.cat([h[:, 0], h.mean(1), h.max(1).values], 1)).squeeze(-1)

def train_model(cols, epochs, seed, tag, n_pos=40, n_neg=40, patience=4, verbose=True):
    torch.manual_seed(seed); np.random.seed(seed)
    tr, va = PatchDS(splits['train'], cols, n_pos, n_neg, seed), PatchDS(splits['val'], cols, 24, 24, 0)
    model = DenseGCN2(len(cols) + 2).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best, hist, path, bad = -1, [], f'{GNN_DIR}/models/gnn_v3_{tag}.pt', 0
    for ep in range(1, epochs + 1):
        tr.epoch = ep; model.train(); tl = n = 0
        for x, A, y in loader(tr, True):
            x, A, y = x.to(device), A.to(device), y.to(device)
            x = x + 0.05 * torch.randn_like(x)                      # input noise
            loss = F_.binary_cross_entropy_with_logits(model(x, A), y)
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tl += loss.item() * len(y); n += len(y)
        sched.step()
        auc, ap = patch_eval([model], va)
        hist.append({'epoch': ep, 'train_loss': tl / n, 'val_auroc': auc, 'val_auprc': ap})
        flag = ''
        if ap > best:
            best, bad, flag = ap, 0, ' <- best'
            torch.save({'model': model.state_dict(), 'cols': list(cols), 'epoch': ep}, path)
        else:
            bad += 1
        if verbose or bad >= patience:
            print(f'[{tag}] epoch {ep:2d} | train loss {tl/n:.4f} | val AUROC {auc:.3f} AUPRC {ap:.3f}{flag}', flush=True)
        if bad >= patience:
            print(f'[{tag}] early stop (best val AUPRC {best:.3f})'); break
    model.load_state_dict(torch.load(path, map_location=device, weights_only=False)['model']); model.eval()
    return model, pd.DataFrame(hist)

In [ ]:
# ── Cell 7: Train the ensemble (full feature set) and report patch-level metrics ──
EPOCHS, N_SEEDS = 40, 3
ENS, hists = [], []
for sd in range(N_SEEDS):
    m, h = train_model(ALL_COLS, EPOCHS, SEED + sd, f'full_s{sd}', verbose=(sd == 0))
    ENS.append(m); hists.append(h); h.to_csv(f'{GNN_DIR}/models/gnn_v3_full_s{sd}_history.csv', index=False)

test_ds = PatchDS(splits['test'], ALL_COLS, 24, 24, 0)
print('\nPATCH-LEVEL (balanced 1:1 sampling, chance = 0.5)')
for i, m in enumerate(ENS): print(f'  seed {i}: test AUROC {patch_eval([m], test_ds)[0]:.3f}')
t_auc, t_ap = patch_eval(ENS, test_ds)
print(f'  ENSEMBLE of {N_SEEDS}: test AUROC {t_auc:.3f}  AUPRC {t_ap:.3f}')
print('  (previous model on its own, weaker split: test AUROC 0.668)')

[full_s0] epoch  1 | train loss 0.6349 | val AUROC 0.714 AUPRC 0.709 <- best
[full_s0] epoch  2 | train loss 0.6205 | val AUROC 0.719 AUPRC 0.715 <- best
[full_s0] epoch  3 | train loss 0.6189 | val AUROC 0.722 AUPRC 0.717 <- best
[full_s0] epoch  4 | train loss 0.6135 | val AUROC 0.719 AUPRC 0.716
[full_s0] epoch  5 | train loss 0.6120 | val AUROC 0.727 AUPRC 0.721 <- best
[full_s0] epoch  6 | train loss 0.6095 | val AUROC 0.727 AUPRC 0.719
[full_s0] epoch  7 | train loss 0.6075 | val AUROC 0.729 AUPRC 0.722 <- best
[full_s0] epoch  8 | train loss 0.6070 | val AUROC 0.723 AUPRC 0.718
[full_s0] epoch  9 | train loss 0.6038 | val AUROC 0.728 AUPRC 0.721
[full_s0] epoch 10 | train loss 0.5987 | val AUROC 0.723 AUPRC 0.720
[full_s0] epoch 11 | train loss 0.5984 | val AUROC 0.726 AUPRC 0.718
[full_s0] early stop (best val AUPRC 0.722)
[full_s1] epoch 10 | train loss 0.5981 | val AUROC 0.728 AUPRC 0.719
[full_s1] early stop (best val AUPRC 0.725)
[full_s2] epoch 12 | train loss 0.5967 | val

In [ ]:
# ── Cell 8 (2b): Residue-level baselines trained on the SAME training surfaces ─
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingClassifier

def surface_arrays(sid):
    z = np.load(f'{FEAT}/{sid}.npz')
    F = z['feats'].astype(np.float32); names = list(z['res_names']); n_res = len(names)
    rt = residue_table(z['atom_xyz'], z['atom_resid'], names, z['near'], F, n_res)
    y = residue_labels(z['atom_resid'], z['atom_iface'], n_res)
    return z, F, rt, y

X1, X2, Y = [], [], []
for sid in splits['train']:
    _, _, rt, y = surface_arrays(sid); m = rt['n_vert'] >= 1
    X1.append(rt['B1'][m]); X2.append(rt['B2'][m]); Y.append(y[m])
X1, X2, Y = np.concatenate(X1), np.concatenate(X2), np.concatenate(Y)
if len(Y) > 400000:
    sel = np.random.default_rng(SEED).choice(len(Y), 400000, replace=False); X1, X2, Y = X1[sel], X2[sel], Y[sel]
print(f'baseline training rows: {len(Y)}  (interface fraction {Y.mean():.3f})')

BASE_MODELS = {
    'lr_comp':      (make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=0.5)).fit(X1, Y), 'B1'),
    'hgb_comp':     (HistGradientBoostingClassifier(max_iter=300, learning_rate=0.08, random_state=SEED).fit(X1, Y), 'B1'),
    'hgb_surface':  (HistGradientBoostingClassifier(max_iter=300, learning_rate=0.08, random_state=SEED).fit(X2, Y), 'B2'),
}
print('Baselines trained: lr_comp (composition+burial), hgb_comp (same, boosted), hgb_surface (+ averaged surface features, no message passing)')

baseline training rows: 195413  (interface fraction 0.190)
Baselines trained: lr_comp (composition+burial), hgb_comp (same, boosted), hgb_surface (+ averaged surface features, no message passing)


In [ ]:
# ── Cell 9 (2b): Scoring utilities; choose post-processing on VAL; compare on held-out TEST complexes ──
from scipy.spatial import cKDTree

@torch.no_grad()
def score_vertices(models, cols, V, Nr, F, batch=1024):
    for m in models: m.eval()
    tree, Fc, out = cKDTree(V), F[:, cols], np.zeros(len(V), np.float32)
    for s in range(0, len(V), batch):
        xs, As = zip(*(patch_arrays2(V, Nr, Fc, tree, i) for i in range(s, min(s + batch, len(V)))))
        xb, Ab = torch.from_numpy(np.stack(xs)).to(device), torch.from_numpy(np.stack(As)).to(device)
        out[s:s + len(xs)] = torch.stack([torch.sigmoid(m(xb, Ab)) for m in models]).mean(0).cpu().numpy()
    return out

CONFIGS = [(how, a) for how in ('top3', 'mean', 'max') for a in (0.0, 0.3, 0.5, 0.7)]

def res_metrics(score, y, mask):
    yy, ss = y[mask], score[mask]
    if yy.sum() == 0 or yy.sum() == len(yy): return np.nan, np.nan
    return roc_auc_score(yy, ss), average_precision_score(yy, ss)

def eval_surface_set(sids, models, cols, n_max, with_baselines=False, seed=0):
    rng = np.random.default_rng(seed); sids = list(rng.permutation(sids)[:n_max]); rows = []
    for sid in sids:
        z, F, rt, y = surface_arrays(sid)
        V, Nr = z['verts'], z['normals'].astype(np.float32); near, rid = z['near'], z['atom_resid']
        vs = score_vertices(models, cols, V, Nr, F); mask = rt['n_vert'] >= 3
        raw = {h: aggregate_vertices(vs, near, rid, len(y), h) for h in ('top3', 'mean', 'max')}
        for how, a in CONFIGS:
            auc, ap = res_metrics(smooth_residues(raw[how], rt['centroid'], a), y, mask)
            rows.append(dict(sid=sid, method=f'gnn_{how}_a{a}', auroc=auc, auprc=ap))
        if with_baselines:
            for name, (mdl, tab) in BASE_MODELS.items():
                auc, ap = res_metrics(mdl.predict_proba(rt[tab])[:, 1], y, mask)
                rows.append(dict(sid=sid, method=name, auroc=auc, auprc=ap))
    return pd.DataFrame(rows)

N_EVAL = 80
t0 = time.time()
val_df = eval_surface_set(splits['val'], ENS, ALL_COLS, N_EVAL)
val_tab = val_df.groupby('method')[['auroc', 'auprc']].mean().sort_values('auroc', ascending=False)
BEST = max(CONFIGS, key=lambda c: val_tab.loc[f'gnn_{c[0]}_a{c[1]}', 'auroc'])
print(f'Post-processing chosen on {N_EVAL} VALIDATION surfaces ({(time.time()-t0)/60:.1f} min):')
print(val_tab.head(6).round(3).to_string()); print('-> BEST post-processing:', BEST)

test_df = eval_surface_set(splits['test'], ENS, ALL_COLS, N_EVAL, with_baselines=True, seed=1)
keep = {f'gnn_{BEST[0]}_a{BEST[1]}': 'GNN ensemble + best post-proc', 'gnn_top3_a0.0': 'GNN ensemble (plain top3)',
        'lr_comp': 'baseline: LR composition+burial', 'hgb_comp': 'baseline: boosted composition+burial',
        'hgb_surface': 'baseline: boosted + averaged surface feats (no message passing)'}
tt = test_df[test_df.method.isin(keep)].groupby('method')[['auroc', 'auprc']].mean().rename(index=keep)
print(f'\nRESIDUE-LEVEL on {N_EVAL} HELD-OUT TEST surfaces (exposed residues; mean over surfaces):')
print(tt.sort_values('auroc', ascending=False).round(3).to_string())
piv = test_df.pivot(index='sid', columns='method', values='auroc')
g = f'gnn_{BEST[0]}_a{BEST[1]}'
for b in ('lr_comp', 'hgb_comp', 'hgb_surface'):
    print(f'GNN beats {b:<12} on {(piv[g] > piv[b]).sum()} / {piv[[g, b]].dropna().shape[0]} test surfaces')

Post-processing chosen on 80 VALIDATION surfaces (2.6 min):
               auroc  auprc
method                     
gnn_max_a0.3   0.732  0.472
gnn_top3_a0.3  0.732  0.471
gnn_top3_a0.5  0.732  0.463
gnn_max_a0.5   0.732  0.462
gnn_mean_a0.5  0.724  0.457
gnn_top3_a0.7  0.722  0.449
-> BEST post-processing: ('max', 0.3)

RESIDUE-LEVEL on 80 HELD-OUT TEST surfaces (exposed residues; mean over surfaces):
                                                                 auroc  auprc
method                                                                       
GNN ensemble + best post-proc                                    0.759  0.535
GNN ensemble (plain top3)                                        0.751  0.531
baseline: boosted + averaged surface feats (no message passing)  0.660  0.418
baseline: boosted composition+burial                             0.614  0.373
baseline: LR composition+burial                                  0.609  0.363
GNN beats lr_comp      on 68 / 79 test surfaces


In [ ]:
# ── Cell 10 (2b): Benchmark targets - GNN vs baselines, with a spatial-null permutation test ──
from Bio.PDB.SASA import ShrakeRupley

def load_target(path):
    s = PDB.PDBParser(QUIET=True).get_structure('t', path)[0]
    xyz, el, rs, nm, rk = [], [], [], [], []
    for ch in s:
        for r in ch:
            if r.id[0] != ' ': continue
            for a in r:
                e = (a.element or a.get_name()[0]).strip().upper()
                if e == 'H' or a.get_altloc() not in (' ', 'A'): continue
                xyz.append(a.coord); el.append(e); rs.append(r.get_resname()); nm.append(a.get_name().strip())
                rk.append(f'{ch.id}:{r.id[1]}{r.id[2].strip()}')
    return np.array(xyz), el, rs, nm, rk

def residue_sasa(path):
    st = PDB.PDBParser(QUIET=True).get_structure('t', path)
    s0 = PDB.Structure.Structure('s'); s0.add(st[0].copy())      # model 0 only (NMR files hold ~20 models)
    ShrakeRupley(probe_radius=1.4, n_points=100).compute(s0, level='R')
    return {f'{r.get_parent().id}:{r.id[1]}{r.id[2].strip()}': r.sasa for r in s0[0].get_residues() if r.id[0] == ' '}

BT = {}
for key, v in bench.items():
    gdir = f'{GOLD}/{key}_{v["pdb_id"]}'
    if not os.path.exists(f'{gdir}/interface_residues.json'): continue
    info = json.load(open(f'{gdir}/interface_residues.json'))
    xyz, el, rs, nm, rk = load_target(f'{gdir}/target.pdb')
    uniq = list(dict.fromkeys(rk)); ridx = {k: i for i, k in enumerate(uniq)}
    a_resid = np.array([ridx[k] for k in rk])
    names = [None] * len(uniq)
    for r_, i_ in zip(rs, a_resid):
        if names[i_] is None: names[i_] = r_
    o = featurize_surface2(xyz, el, rs, nm)
    rt = residue_table(xyz, a_resid, names, o['near'], o['feats'], len(uniq))
    sasa = residue_sasa(f'{gdir}/target.pdb'); sa = np.array([sasa.get(k, 0.0) for k in uniq])
    posset = set(info['interface_consensus'])
    BT[key] = dict(pdb=v['pdb_id'], target=v['target'], uniq=uniq, o=o, a_resid=a_resid, rt=rt, names=names,
                   sasa=sa, exposed=sa > 10.0, pos=np.array([k in posset for k in uniq]).astype(int))
print(f'{len(BT)} benchmark targets featurised')

def target_methods(T, models, cols, best):
    o, rt, n = T['o'], T['rt'], len(T['uniq'])
    vs = score_vertices(models, cols, o['verts'], o['normals'], o['feats'])
    raw = {h: aggregate_vertices(vs, o['near'], T['a_resid'], n, h) for h in ('top3', 'mean', 'max')}
    hyd = np.array([(KD_SCALE.get(r, 0.0) + 1) / 2 for r in T['names']])
    M = {'gnn_post': smooth_residues(raw[best[0]], rt['centroid'], best[1]), 'gnn_top3': raw['top3'],
         'sasa': T['sasa'], 'sasa_x_hydro': T['sasa'] * hyd}
    for name, (mdl, tab) in BASE_MODELS.items(): M[name] = mdl.predict_proba(rt[tab])[:, 1]
    return M

def spatial_null(score, T, n=1000, seed=0):
    # Null: a random CONTIGUOUS surface patch with the same number of residues as the true interface.
    # Tests whether a method finds THIS interface, not just "some compact exposed patch".
    rng = np.random.default_rng(seed); idx = np.where(T['exposed'])[0]; y = T['pos'][idx]; k = int(y.sum())
    if k == 0 or k == len(idx): return np.nan, np.nan, np.nan
    obs = roc_auc_score(y, score[idx]); tree = cKDTree(T['rt']['centroid'][idx]); cen = T['rt']['centroid'][idx]; null = []
    for _ in range(n):
        nn_ = np.atleast_1d(tree.query(cen[rng.integers(len(idx))], k=k)[1])
        yy = np.zeros(len(idx), int); yy[nn_] = 1; null.append(roc_auc_score(yy, score[idx]))
    return obs, float(np.mean(null)), (1 + int(np.sum(np.array(null) >= obs))) / (n + 1)

rows, resid_scores = [], {}
for key, T in BT.items():
    M = target_methods(T, ENS, ALL_COLS, BEST); ex = T['exposed']
    resid_scores[key] = {k: dict(zip(T['uniq'], map(float, M[k]))) for k in ('gnn_post', 'gnn_top3')}
    for name, sc in M.items():
        auc, ap = res_metrics(sc, T['pos'], ex)
        yy = T['pos'][ex]; top = np.argsort(-sc[ex])[:int(yy.sum())]
        obs, nullm, p = spatial_null(sc, T, n=1000) if name in ('gnn_post', 'hgb_surface', 'hgb_comp', 'sasa_x_hydro') else (np.nan,) * 3
        rows.append(dict(id=int(key), pdb=T['pdb'], target=T['target'], method=name, auroc=auc, auprc=ap,
                         r_precision=float(yy[top].mean()), null_auroc=nullm, p_spatial=p, prevalence=float(yy.mean())))
    json.dump(resid_scores[key], open(f'{RESULTS}/gnn_scores_v3/{key}_{T["pdb"]}.json', 'w'))
    print(f'[{key}] {T["pdb"]} done', flush=True)

ev = pd.DataFrame(rows); ev.to_csv(f'{RESULTS}/gnn_v3_benchmark_eval.csv', index=False)
print('\nMean over the benchmark targets (exposed residues):')
print(ev.groupby('method')[['auroc', 'auprc', 'r_precision']].mean().sort_values('auroc', ascending=False).round(3).to_string())
pv = ev.pivot(index='target', columns='method', values='auroc')
for b in ('hgb_surface', 'hgb_comp', 'lr_comp', 'sasa_x_hydro'):
    print(f'gnn_post beats {b:<13} on {(pv["gnn_post"] > pv[b]).sum()} / {len(pv)} targets')
sp = ev[ev.method == 'gnn_post']
print(f'\nSpatial-null test (gnn_post): significant at p<0.05 on {(sp.p_spatial < 0.05).sum()} / {len(sp)} targets; '
      f'mean null AUROC {sp.null_auroc.mean():.3f} vs observed {sp.auroc.mean():.3f}')
print('\nPer-target AUROC:'); print(pv[['gnn_post', 'gnn_top3', 'hgb_surface', 'hgb_comp', 'lr_comp', 'sasa_x_hydro']].round(2).to_string())
print('\nPer-target spatial-null p (gnn_post):'); print(sp.set_index('target')[['auroc', 'null_auroc', 'p_spatial']].round(3).to_string())

20 benchmark targets featurised
[4] 1JSU done
[5] 1YCR done
[6] 1JM7 done
[7] 1N0W done
[8] 1AXC done
[10] 2XA0 done
[11] 1G5J done
[12] 3EZQ done
[13] 1NW9 done
[20] 4G0N done
[22] 1KJY done
[24] 1J2J done
[26] 1FOS done
[27] 1NKP done
[28] 1KDX done
[30] 4GQ6 done
[33] 1LM8 done
[34] 2FLU done
[37] 1JDH done
[38] 1T01 done

Mean over the benchmark targets (exposed residues):
              auroc  auprc  r_precision
method                                 
gnn_post      0.825  0.572        0.519
gnn_top3      0.814  0.578        0.521
hgb_surface   0.675  0.417        0.379
hgb_comp      0.615  0.324        0.293
lr_comp       0.590  0.313        0.303
sasa_x_hydro  0.565  0.292        0.276
sasa          0.522  0.255        0.244
gnn_post beats hgb_surface   on 19 / 20 targets
gnn_post beats hgb_comp      on 19 / 20 targets
gnn_post beats lr_comp       on 19 / 20 targets
gnn_post beats sasa_x_hydro  on 20 / 20 targets

Spatial-null test (gnn_post): significant at p<0.05 on 15 / 20 targ

In [ ]:
# ── Cell 11 (2b): Feature ablation (single seed, short schedule so variants are comparable) ──
RUN_ABLATION, ABL_EPOCHS = True, 15
if RUN_ABLATION:
    G = FEATURE_GROUPS
    drop = lambda *gs: [c for c in ALL_COLS if c not in {x for g_ in gs for x in G[g_]}]
    VARIANTS = {'full': ALL_COLS, 'no_shape': drop('shape'), 'no_aa': drop('aa'),
                'no_electro_smooth_burial': drop('electro_smooth', 'burial'), 'shape_only': G['shape']}
    abl = []
    for name, cols in VARIANTS.items():
        m, h = train_model(cols, ABL_EPOCHS, SEED, f'abl_{name}', verbose=False)
        vd = eval_surface_set(splits['val'], [m], cols, 50)
        v_auc = vd[vd.method == 'gnn_top3_a0.0'].auroc.mean()
        b_auc = np.nanmean([res_metrics(aggregate_vertices(score_vertices([m], cols, T['o']['verts'], T['o']['normals'], T['o']['feats']),
                            T['o']['near'], T['a_resid'], len(T['uniq']), 'top3'), T['pos'], T['exposed'])[0] for T in BT.values()])
        abl.append(dict(variant=name, n_features=len(cols), val_patch_auroc=float(h.val_auroc.max()),
                        val_residue_auroc=float(v_auc), benchmark_auroc=float(b_auc)))
        print(abl[-1], flush=True)
    abl = pd.DataFrame(abl); abl.to_csv(f'{RESULTS}/gnn_v3_ablation.csv', index=False)
    print('\nFEATURE ABLATION (15-epoch single models):'); print(abl.round(3).to_string(index=False))

[abl_full] epoch 11 | train loss 0.5949 | val AUROC 0.728 AUPRC 0.723
[abl_full] early stop (best val AUPRC 0.723)
{'variant': 'full', 'n_features': 31, 'val_patch_auroc': 0.7290009913297172, 'val_residue_auroc': 0.702074433109622, 'benchmark_auroc': 0.8032847194119052}
[abl_no_shape] epoch 15 | train loss 0.5913 | val AUROC 0.720 AUPRC 0.714
[abl_no_shape] early stop (best val AUPRC 0.718)
{'variant': 'no_shape', 'n_features': 29, 'val_patch_auroc': 0.7234810568704799, 'val_residue_auroc': 0.7044289080094001, 'benchmark_auroc': 0.8162111570361285}
[abl_no_aa] epoch  9 | train loss 0.6153 | val AUROC 0.712 AUPRC 0.709
[abl_no_aa] early stop (best val AUPRC 0.711)
{'variant': 'no_aa', 'n_features': 10, 'val_patch_auroc': 0.7154868687335635, 'val_residue_auroc': 0.6946455985126693, 'benchmark_auroc': 0.791431775169184}
[abl_no_electro_smooth_burial] epoch  8 | train loss 0.6138 | val AUROC 0.720 AUPRC 0.712
[abl_no_electro_smooth_burial] early stop (best val AUPRC 0.714)
{'variant': 'no_

In [ ]:
import glob
fixed = []
for p in glob.glob(f'{GOLD}/*/*.pdb'):
    lines = open(p).readlines()
    if sum(l.startswith('MODEL') for l in lines) > 1:
        out = []
        for l in lines:
            out.append(l)
            if l.startswith('ENDMDL'): break
        out.append('END\n'); open(p, 'w').writelines(out); fixed.append(p)
print(len(fixed), 'files cut to model 0:', sorted({p.split('/')[-2] for p in fixed}))

6 files cut to model 0: ['28_1KDX', '6_1JM7']


In [ ]:
# ── Cell 12: Save manifest ────────────────────────────────────────────────────
import Bio, sklearn
json.dump({'created': time.strftime('%Y-%m-%d %H:%M:%S'), 'seed': SEED, 'torch': torch.__version__,
           'biopython': Bio.__version__, 'sklearn': sklearn.__version__, 'n_features': N_FEAT, 'n_seeds': N_SEEDS,
           'epochs': EPOCHS, 'excluded_entries': len(EXCLUDED), 'best_postproc': list(BEST),
           'surfaces': {k: len(v) for k, v in splits.items()}, 'patch_test_auroc': t_auc, 'patch_test_auprc': t_ap,
           'benchmark_mean_auroc': ev.groupby('method').auroc.mean().round(4).to_dict()},
          open(f'{MANIFEST}/phase2b_manifest.json', 'w'), indent=2)
print('Saved manifests/phase2b_manifest.json. Bring the Cell 7, 9, 10 and 11 outputs back for review.')

Saved manifests/phase2b_manifest.json. Bring the Cell 7, 9, 10 and 11 outputs back for review.
